In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

In [34]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import plotly.express as px
from sklearn.model_selection import GridSearchCV


In [3]:
df = pd.read_csv(r'/content/loan_approval_data (1) (1).csv')
df.head()

,id,age,income,home_ownership,emplyment_length,loan_intent,loan_amount,loan_interest_rate,loan_income_ratio,payment_default_on_file,credit_history_length,loan_approval_status,max_allowed_loan
0,35437,21.0,12000,OWN,0,EDUCATION,15000,6.99,0.12,N,4,0,-2426900
1,53756,21.0,13200,OWN,2,EDUCATION,25000,16.77,0.19,Y,3,0,-111739
2,42205,23.0,9600,RENT,5,MEDICAL,30000,12.42,0.31,N,3,0,-89000
3,19180,40.0,182004,RENT,3,EDUCATION,35000,8.00,0.19,N,11,0,35000
4,28072,40.0,90000,MORTGAGE,3,HOMEIMPROVEMENT,35000,12.42,0.39,N,14,0,35000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58645 entries, 0 to 58644
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       58645 non-null  int64  
 1   age                      58639 non-null  float64
 2   income                   58645 non-null  int64  
 3   home_ownership           58645 non-null  object 
 4   emplyment_length         58645 non-null  int64  
 5   loan_intent              58645 non-null  object 
 6   loan_amount              58645 non-null  int64  
 7   loan_interest_rate       58634 non-null  float64
 8   loan_income_ratio        58645 non-null  float64
 9   payment_default_on_file  58640 non-null  object 
 10  credit_history_length    58645 non-null  int64  
 11  loan_approval_status     58645 non-null  int64  
 12  max_allowed_loan         58645 non-null  int64  
dtypes: float64(3), int64(7), object(3)
memory usage: 5.8+ MB


In [5]:
df.isnull().sum()

,0
id,0
age,6
income,0
home_ownership,0
emplyment_length,0
loan_intent,0
loan_amount,0
loan_interest_rate,11
loan_income_ratio,0
payment_default_on_file,5


In [6]:
# Fill numeric columns with median
df['age'].fillna(df['age'].median(), inplace=True)
df['loan_interest_rate'].fillna(df['loan_interest_rate'].median(), inplace=True)

# Fill categorical column with mode
df['payment_default_on_file'].fillna(df['payment_default_on_file'].mode()[0], inplace=True)

/tmp/ipykernel_630/2077169524.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['age'].fillna(df['age'].median(), inplace=True)
/tmp/ipykernel_630/2077169524.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

In [7]:
df.isnull().sum()

,0
id,0
age,0
income,0
home_ownership,0
emplyment_length,0
loan_intent,0
loan_amount,0
loan_interest_rate,0
loan_income_ratio,0
payment_default_on_file,0


In [12]:
df_model = df.drop(columns=['id', 'max_allowed_loan'])


In [15]:
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

In [16]:
df_model.fillna(df_model.median(), inplace=True)


In [17]:
X = df_model.drop(columns=['loan_approval_status'])
y = df_model['loan_approval_status']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [21]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


In [22]:
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")


Train: 46916 | Test: 11729


In [24]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_sc, y_train)
lr_pred = lr_model.predict(X_test_sc)
lr_prob = lr_model.predict_proba(X_test_sc)[:, 1]
print("✅ Logistic Regression")

✅ Logistic Regression


In [25]:
print(f"   Accuracy : {accuracy_score(y_test, lr_pred):.4f}")
print(f"   ROC-AUC  : {roc_auc_score(y_test, lr_prob):.4f}\n")

   Accuracy : 0.8902
   ROC-AUC  : 0.8714



In [26]:
nb_model = GaussianNB()
nb_model.fit(X_train_sc, y_train)
nb_pred = nb_model.predict(X_test_sc)
nb_prob = nb_model.predict_proba(X_test_sc)[:, 1]
print("✅ Naïve Bayes")

✅ Naïve Bayes


In [27]:
print(f"   Accuracy : {accuracy_score(y_test, nb_pred):.4f}")
print(f"   ROC-AUC  : {roc_auc_score(y_test, nb_prob):.4f}\n")



   Accuracy : 0.8493
   ROC-AUC  : 0.8548



In [28]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_sc, y_train)
knn_pred = knn_model.predict(X_test_sc)
knn_prob = knn_model.predict_proba(X_test_sc)[:, 1]
print("✅ KNN")

✅ KNN


In [29]:
print(f"   Accuracy : {accuracy_score(y_test, knn_pred):.4f}")
print(f"   ROC-AUC  : {roc_auc_score(y_test, knn_prob):.4f}")

   Accuracy : 0.9142
   ROC-AUC  : 0.8552


In [31]:
cm = confusion_matrix(y_test, lr_pred)
fig = px.imshow(cm, text_auto=True,
                x=['Rejected', 'Approved'],
                y=['Rejected', 'Approved'],
                title='Confusion Matrix – Logistic Regression')
fig.show()

In [33]:
print(classification_report(y_test, lr_pred,
      target_names=['Rejected', 'Approved']))


              precision    recall  f1-score   support

    Rejected       0.90      0.97      0.94     10059
    Approved       0.72      0.38      0.50      1670

    accuracy                           0.89     11729
   macro avg       0.81      0.68      0.72     11729
weighted avg       0.88      0.89      0.88     11729



In [35]:
param_grid = {'n_neighbors': [3, 5, 7, 11]}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid.fit(X_train_sc, y_train)
print("Best:", grid.best_params_)

Best: {'n_neighbors': 11}
